## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks demonstrates how to access the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: [terraclimate- overview](https://planetarycomputer.microsoft.com/dataset/terraclimate#overview) 

## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process.

In [ ]:
from snowflake.snowpark.context import get_active_session
import time
import os

session = get_active_session()

# Menambahkan numpy==1.26.4 di urutan teratas untuk mencegah bentrok NumPy 2.0
requirements = """
numpy==1.26.4
rioxarray==0.17.0
pystac==1.11.0
pystac_client==0.9.0
planetary_computer==1.0.0
odc-stac==0.3.10
shapely==2.1.1
rasterio==1.4.3
tqdm==4.66.5
pandas==2.3.0
xarray==2025.3.1
matplotlib==3.10.3
geopandas==1.1.1
seaborn==0.13.2
scikit-learn==1.5.2
netCDF4==1.7.2
adlfs==2025.8.0
zarr==2.17.2
dask==2024.10.0
xarray[complete]
numcodecs==0.12.1
"""

with open('/tmp/requirements.txt', 'w') as f:
    f.write(requirements)

print("File requirements.txt berhasil dibuat di /tmp")

session.sql("""
    PUT file:///tmp/requirements.txt
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File ter-upload ke environment Snowflake!")

start_time = time.time()
print("Start Install Package ...")

!pip install uv
!uv pip install -r /tmp/requirements.txt

elapsed_time = time.time() - start_time
print(f"\n✓ Instalasi Package Selesai 100%!")
print(f"  Waktu yang dihabiskan: {elapsed_time:.1f} detik ({elapsed_time/60:.1f} menit)")

In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr
from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

# Definisikan lokasi stage utama Anda
STAGE_PATH = "@SNOWFLAKE_LEARNING_DB.PUBLIC.EXTERNAL_DATA_STAGE"
print("Setup berhasil. Lingkungan siap digunakan.")

## Extracting TerraClimate Data Using API Calls

The API-based method allows us to efficiently access **TerraClimate** data for specific regions and time periods through the [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/), ensuring scalability and reproducibility of the process.

Through the API, we can extract climate variables such as **Potential Evapotranspiration (PET)**, which represents the atmospheric demand for water. This variable provides important insights into surface moisture balance and helps improve the accuracy of water quality modeling.

This approach ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis.


### Loading and Mapping TerraClimate Data

This section demonstrates how **TerraClimate climate variables**, such as **Potential Evapotranspiration (PET)**, are loaded and mapped to sampling locations.

- The **load_terraclimate_dataset** function opens the TerraClimate Zarr/NetCDF dataset from the Microsoft Planetary Computer, handling storage options automatically.
- The **filterg** function filters the dataset for the desired time range (2011–2015) and the spatial extent corresponding to the study region. The resulting data is converted into a pandas DataFrame with standardized column names.
- The **assign_nearest_climate** function maps each sampling location to its **nearest TerraClimate grid point** using a KD-tree and assigns the climate variable values corresponding to the closest timestamp.

This workflow ensures efficient, reproducible retrieval of climate variables, while allowing participants to work with pre-extracted CSV files for faster benchmarking and analysis.


In [2]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [3]:
# --- Filtering function (kept identical) ---
def filterg_multi(ds, var_list):
    """Memotong data menggunakan Slicing Sumbu (Axis Slicing) untuk kecepatan maksimal."""
    ds_2011_2015 = ds[var_list].sel(time=slice("2011-01-01", "2015-12-31"))

    print("Memotong Area Bounding Box dengan Slicing Metadata...")
    # 2. Pemotongan Malas (Sangat Cepat). 
    # Catatan: TerraClimate Lat turun (max ke min), Lon naik (min ke max)
    ds_sa = ds_2011_2015.sel(
        lat=slice(-21.72, -35.18), 
        lon=slice(14.97, 32.79)
    )
    
    # 3. KUNCI UTAMA: Tarik semua bongkahan kecil ini ke RAM seketika!
    print("Mengunduh bongkahan data ke memori (Menghindari Token Timeout)...")
    ds_sa = ds_sa.compute() 
    print("✓ Unduhan selesai!")

    df_var_append = []
    
    # Looping sekarang terjadi murni di dalam RAM Anda (Tanpa koneksi internet lagi)
    for i in tqdm(range(len(ds_sa.time)), desc="Mengonversi ke DataFrame"):
        df_var = ds_sa.isel(time=i).to_dataframe().reset_index()
        
        # Bersihkan NaN bawaan dari kotak koordinat
        df_var_clean = df_var.dropna(subset=['lat', 'lon', var_list[0]])
        df_var_append.append(df_var_clean)

    # Gabungkan semua bulan
    df_var_final = pd.concat(df_var_append, ignore_index=True)
    df_var_final['time'] = df_var_final['time'].astype(str)
    
    # Pemetaan kolom sesuai format EY
    df_var_final = df_var_final.rename(columns={"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"})

    return df_var_final


In [4]:
# --- Climate variable assignment function (unchanged logic) ---
def assign_nearest_climate_multi(sa_df, climate_df, var_list):
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    # Siapkan kamus (dictionary) untuk menyimpan hasil ke-14 fitur
    climate_values = {f"terra_{var}": [] for var in var_list}

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {len(var_list)} variables"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        # Jika titik tidak ditemukan, isi NaN untuk ke-14 variabel
        if subset.empty:
            for var in var_list:
                climate_values[f"terra_{var}"].append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        
        # Tarik semua nilai variabel di index waktu terdekat
        for var in var_list:
            climate_values[f"terra_{var}"].append(subset.loc[nearest_idx, var])

    output_df = pd.DataFrame(climate_values)
    return output_df

### Extracting features for the training dataset

In [5]:
print("Mengunduh data training dari Stage...")
session.file.get(f"{STAGE_PATH}/water_quality_training_dataset.csv", "/tmp/")

# 3. Baca dengan pandas dari direktori /tmp/
Water_Quality_df = pd.read_csv("/tmp/water_quality_training_dataset.csv")
display(Water_Quality_df.head(5))

In [6]:
Water_Quality_df.shape

In [7]:
# Daftar seluruh parameter TerraClimate (14 Variabel)
all_terra_vars = ['pet', 'ppt', 'tmax', 'tmin', 'vap', 'srad', 'ws', 'aet', 'q', 'def', 'soil', 'swe', 'pdsi', 'vpd']

# Load TerraClimate dataset
ds = load_terraclimate_dataset()

In [ ]:
print("Memotong Area Bounding Box Afrika Selatan...")
tc_parameters_multi = filterg_multi(ds, all_terra_vars)

# Petakan 14 variabel ke data Train
print("\nMengekstrak 14 Variabel untuk Data Train...")
Terraclimate_training_df = assign_nearest_climate_multi(Water_Quality_df, tc_parameters_multi, all_terra_vars)

In [8]:
Terraclimate_training_df['Latitude'] = Water_Quality_df['Latitude']
Terraclimate_training_df['Longitude'] = Water_Quality_df['Longitude']
Terraclimate_training_df['Sample Date'] = Water_Quality_df['Sample Date']

# Urutkan kolom agar rapi
terra_cols = [f"terra_{var}" for var in all_terra_vars]
final_cols = ['Latitude', 'Longitude', 'Sample Date'] + terra_cols
Terraclimate_training_df = Terraclimate_training_df[final_cols]

In [ ]:
display(Terraclimate_training_df)

In [9]:
STAGE_PATH = "@SNOWFLAKE_LEARNING_DB.PUBLIC.EXTERNAL_DATA_STAGE"

# Simpan sebagai Parquet ke penyimpanan memori lokal sementara (/tmp)
Terraclimate_training_df.to_parquet('/tmp/train_terraclimate_features.parquet', index=False)

# Unggah dari /tmp langsung ke EXTERNAL_DATA_STAGE Anda
session.file.put(
    "file:///tmp/train_terraclimate_features.parquet", 
    STAGE_PATH, 
    auto_compress=False, 
    overwrite=True
)
print("✓ Train TerraClimate berhasil disimpan ke EXTERNAL_DATA_STAGE!")

### Extracting features for the validation dataset

In [10]:
print("Mengunduh data testing dari Stage...")
session.file.get(f"{STAGE_PATH}/submission_template.csv", "/tmp/")

Validation_df = pd.read_csv("/tmp/submission_template.csv")
display(Validation_df.head(5))

In [11]:
Validation_df.shape

In [12]:
# Load TerraClimate dataset, filter (time,region,parameter), filter for nearest parameter values
print("\nMengekstrak 14 Variabel untuk Data Test...")
Terraclimate_validation_df = assign_nearest_climate_multi(Validation_df, tc_parameters_multi, all_terra_vars)

Terraclimate_validation_df['Latitude'] = Validation_df['Latitude']
Terraclimate_validation_df['Longitude'] = Validation_df['Longitude']
Terraclimate_validation_df['Sample Date'] = Validation_df['Sample Date']
Terraclimate_validation_df = Terraclimate_validation_df[final_cols]
Terraclimate_validation_df.to_parquet('/tmp/test_terraclimate_features.parquet', index=False)
display(Terraclimate_validation_df)

In [ ]:

# Unggah dari /tmp langsung ke EXTERNAL_DATA_STAGE Anda
session.file.put(
    "file:///tmp/test_terraclimate_features.parquet", 
    STAGE_PATH, 
    auto_compress=False, 
    overwrite=True
)
print("✓ Validation TerraClimate berhasil disimpan ke EXTERNAL_DATA_STAGE!")

In [14]:
# Preview File
display(Terraclimate_validation_df.head())

In [ ]:
Terraclimate_validation_df.to_csv("/tmp/terraclimate_features_validation.csv",index = False)

In [ ]:
session.sql(f"""
    PUT file:///tmp/terraclimate_features_validation.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()


print("File saved! Refresh the browser to see the files in the sidebar")
